In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from jcopml.pipeline import num_pipe, cat_pipe
from jcopml.utils import save_model, load_model
from jcopml.plot import plot_missing_value
from jcopml.feature_importance import mean_score_decrease

In [2]:
df = pd.read_csv("data/MALratings.csv")
df.head()
df.describe()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20343 entries, 0 to 20342
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Title           20343 non-null  object 
 1   Genres          10016 non-null  object 
 2   Rank            20343 non-null  object 
 3   Popularity      20343 non-null  int64  
 4   Score           14019 non-null  float64
 5   Episodes        20343 non-null  object 
 6   Episode length  20343 non-null  object 
 7   Release Date    5087 non-null   object 
dtypes: float64(1), int64(1), object(6)
memory usage: 1.2+ MB


In [3]:
# 1. Hapus baris yang nilai targetnya NaN
df = df.dropna(subset=['Score'])  # Ganti 'nama_kolom_target' dengan kolom target Anda (misal: 'Score')

# 2. Setelah itu, baru lakukan pembersihan kolom numerik dan train_test_split seperti biasa
numeric_cols = ["Rank", "Popularity", "Episodes"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 3. Lanjutkan dengan train_test_split dan fit model

In [4]:
X = df.drop(columns=[
    "Title",
    "Score"
])

y = df["Score"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((11215, 6), (2804, 6), (11215,), (2804,))

In [5]:
preprocessor = ColumnTransformer([
    ('numeric', num_pipe(), ["Popularity", "Episodes", "Rank"]),
    ('categoric', cat_pipe(encoder='onehot'), ["Genres", "Episode length", "Release Date"]),
])

In [6]:
from xgboost import XGBRegressor
pipeline = Pipeline([
    ('prep', preprocessor),
    ('algo', XGBRegressor(n_jobs=-1, random_state=42, eval_metric='mlogloss'))
])

In [7]:
from sklearn.model_selection import RandomizedSearchCV
from jcopml.tuning import random_search_params as rsp

model = RandomizedSearchCV(pipeline, rsp.xgb_params, cv=5, scoring='r2', n_iter=50, n_jobs=-1, verbose=1, random_state=42)
model.fit(X_train, y_train)

print(model.best_params_)
print(model.score(X_train, y_train), model.best_score_, model.score(X_test, y_test))

Fitting 5 folds for each of 50 candidates, totalling 250 fits
{'algo__colsample_bytree': np.float64(0.8347886858974456), 'algo__gamma': 1, 'algo__learning_rate': np.float64(0.020018663891407017), 'algo__max_depth': 6, 'algo__n_estimators': 179, 'algo__reg_alpha': np.float64(0.0014557961490251987), 'algo__reg_lambda': np.float64(0.23097861492693586), 'algo__subsample': np.float64(0.6387821809211413)}
0.972543383884138 0.9684839849330906 0.9702831597054682
